In [0]:
display(spark.sql("SHOW TABLES IN medical_insurance.silver"))

In [0]:
%sql
SHOW TABLES IN medical_insurance.silver

In [0]:
tables = [row.tableName for row in spark.sql("SHOW TABLES IN medical_insurance.silver").collect()]

for table in tables:
    df = spark.sql(f"SELECT * FROM medical_insurance.silver.{table} LIMIT 5")
    print(f"Table: {table}")
    display(df)

In [0]:
%sql
CREATE OR REPLACE TABLE medical_insurance.gold.patient_analytics AS
WITH patient_base AS (
  SELECT 
    p.patient_id,
    p.patient_name,
    p.gender,
    p.birth_date,
    p.patient_age AS age,
    CASE 
      WHEN p.patient_age < 18 THEN 'Child (0-17)'
      WHEN p.patient_age BETWEEN 18 AND 35 THEN 'Young Adult (18-35)'
      WHEN p.patient_age BETWEEN 36 AND 55 THEN 'Middle Age (36-55)'
      WHEN p.patient_age BETWEEN 56 AND 70 THEN 'Senior (56-70)'
      ELSE 'Elderly (70+)'
    END AS age_group,
    p.blood_type,
    p.governorate,
    p.city
  FROM medical_insurance.silver.patient_silver p
),
visit_stats AS (
  SELECT 
    v.patient_id,
    COUNT(DISTINCT v.visit_id) AS total_visits,
    COUNT(DISTINCT CASE WHEN v.visit_type = 'Outpatient' THEN v.visit_id END) AS outpatient_visits,
    COUNT(DISTINCT CASE WHEN v.visit_type = 'Inpatient' THEN v.visit_id END) AS inpatient_visits,
    COUNT(DISTINCT CASE WHEN v.visit_type = 'Follow-up' THEN v.visit_id END) AS followup_visits,
    COUNT(DISTINCT v.hospital_id) AS hospitals_visited,
    AVG(v.waiting_time) AS avg_waiting_time_minutes,
    SUM(v.total_amount) AS total_visit_cost,
    AVG(v.total_amount) AS avg_cost_per_visit,
    MIN(v.visit_date) AS first_visit_date,
    MAX(v.visit_date) AS last_visit_date,
    COUNT(DISTINCT v.diagnosis_code) AS unique_diagnoses
  FROM medical_insurance.silver.visit_silver v
  GROUP BY v.patient_id
),
claim_stats AS (
  SELECT 
    c.patient_id,
    COUNT(DISTINCT c.claim_id) AS total_claims,
    COUNT(DISTINCT CASE WHEN c.claim_status = 'Approved' THEN c.claim_id END) AS approved_claims,
    COUNT(DISTINCT CASE WHEN c.claim_status = 'Rejected' THEN c.claim_id END) AS rejected_claims,
    SUM(c.claim_amount) AS total_claimed_amount,
    SUM(c.approved_amount) AS total_approved_amount,
    SUM(c.claim_amount - c.approved_amount) AS total_denied_amount,
    AVG(c.approved_amount / NULLIF(c.claim_amount, 0)) AS avg_approval_rate
  FROM medical_insurance.silver.claim_silver c
  GROUP BY c.patient_id
),
feedback_stats AS (
  SELECT 
    f.patient_id,
    COUNT(DISTINCT f.feedback_id) AS feedback_count,
    AVG(f.rating) AS avg_satisfaction_rating,
    MIN(f.rating) AS min_rating,
    MAX(f.rating) AS max_rating
  FROM medical_insurance.silver.patient_feedback_silver f
  GROUP BY f.patient_id
)
SELECT 
  pb.patient_id,
  pb.patient_name,
  pb.gender,
  pb.birth_date,
  pb.age,
  pb.age_group,
  pb.blood_type,
  pb.governorate,
  pb.city,
  
  -- Visit metrics
  COALESCE(vs.total_visits, 0) AS total_visits,
  COALESCE(vs.outpatient_visits, 0) AS outpatient_visits,
  COALESCE(vs.inpatient_visits, 0) AS inpatient_visits,
  COALESCE(vs.followup_visits, 0) AS followup_visits,
  COALESCE(vs.hospitals_visited, 0) AS hospitals_visited,
  ROUND(COALESCE(vs.avg_waiting_time_minutes, 0), 2) AS avg_waiting_time_minutes,
  vs.first_visit_date,
  vs.last_visit_date,
  COALESCE(vs.unique_diagnoses, 0) AS unique_diagnoses,
  
  -- Cost metrics
  ROUND(COALESCE(vs.total_visit_cost, 0), 2) AS total_visit_cost,
  ROUND(COALESCE(vs.avg_cost_per_visit, 0), 2) AS avg_cost_per_visit,
  
  -- Claim metrics
  COALESCE(cs.total_claims, 0) AS total_claims,
  COALESCE(cs.approved_claims, 0) AS approved_claims,
  COALESCE(cs.rejected_claims, 0) AS rejected_claims,
  ROUND(COALESCE(cs.total_claimed_amount, 0), 2) AS total_claimed_amount,
  ROUND(COALESCE(cs.total_approved_amount, 0), 2) AS total_approved_amount,
  ROUND(COALESCE(cs.total_denied_amount, 0), 2) AS total_denied_amount,
  ROUND(COALESCE(cs.avg_approval_rate * 100, 0), 2) AS claim_approval_rate_pct,
  
  -- Satisfaction metrics
  COALESCE(fs.feedback_count, 0) AS feedback_count,
  ROUND(COALESCE(fs.avg_satisfaction_rating, 0), 2) AS avg_satisfaction_rating,
  fs.min_rating,
  fs.max_rating,
  
  -- Risk indicators
  CASE 
    WHEN vs.total_visits > 10 THEN 'High Utilization'
    WHEN vs.total_visits BETWEEN 5 AND 10 THEN 'Medium Utilization'
    WHEN vs.total_visits < 5 THEN 'Low Utilization'
    ELSE 'No Visits'
  END AS utilization_category,
  
  CURRENT_TIMESTAMP() AS created_at
  
FROM patient_base pb
LEFT JOIN visit_stats vs ON pb.patient_id = vs.patient_id
LEFT JOIN claim_stats cs ON pb.patient_id = cs.patient_id
LEFT JOIN feedback_stats fs ON pb.patient_id = fs.patient_id

In [0]:
%sql
-- Display sample records from patient analytics gold table
SELECT *
FROM medical_insurance.gold.patient_analytics
LIMIT 10

In [0]:
%sql
-- Summary statistics from patient analytics
SELECT 
  COUNT(DISTINCT patient_id) AS total_patients,
  age_group,
  gender,
  COUNT(*) AS patient_count,
  ROUND(AVG(total_visits), 2) AS avg_visits_per_patient,
  ROUND(AVG(total_visit_cost), 2) AS avg_total_cost,
  ROUND(AVG(avg_satisfaction_rating), 2) AS avg_satisfaction,
  ROUND(AVG(claim_approval_rate_pct), 2) AS avg_claim_approval_rate,
  utilization_category
FROM medical_insurance.gold.patient_analytics
GROUP BY age_group, gender, utilization_category
ORDER BY age_group, gender

In [0]:
%sql
select count(*) from medical_insurance.gold.patient_analytics